In [1]:
import datetime
import pandas as pd
import numpy as np
from sts.quants.estimators.vol import robust_vol_calc
from sts.strategy.pca.pca_analysis import PCATS

In [2]:
from sts.data import yh_data

In [3]:
yh_data.symbol_table

,SymId,AssetClass,Symbol,Description,YahooSym
0,1,ETF,XLE,Energy Select Sector SPDR Fund,XLE
1,2,ETF,XLF,Financial Select Sector SPDR Fund,XLF
2,3,ETF,XLU,Utilities Select Sector SPDR Fund,XLU
3,4,ETF,XLI,Industrial Select Sector SPDR Fund,XLI
4,5,ETF,GDX,VanEck Gold Miners ETF,GDX
5,6,ETF,XLK,Technology Select Sector SPDR Fund,XLK
6,7,ETF,XLV,Health Care Select Sector SPDR Fund,XLV
7,8,ETF,XLY,Consumer Discretionary Select Sector SPDR Fund,XLY
8,9,ETF,XLP,Consumer Staples Select Sector SPDR Fund,XLP
9,10,ETF,XLB,Materials Select Sector SPDR Fund,XLB


In [4]:
look_back_history = 360 * 2
cov_mat_diff_days = 5
ed = datetime.date.today()  # datetime.date.today()
sd = ed - datetime.timedelta(days=look_back_history)

In [5]:
etf_list = [
    "XLK",
    "XLV",
    "XLE",
    "XLF",
    "XLU",
    "XLP",
    "XLY",
]  # , 'IYR', 'XHB', 'XRT', 'SMH', 'IBB', 'KBE']

In [6]:
price_df = yh_data.get_hist_data_for_symbols(etf_list, sd, ed, column="Close")

In [7]:
pcats = PCATS(price_df, corr_type_percent=True, cov_mat_diff_days=cov_mat_diff_days)

In [8]:
pcats.corr_mat

,XLK,XLV,XLE,XLF,XLU,XLP,XLY
XLK,1.000000,0.581075,0.182420,0.673378,0.432006,0.532418,0.835308
XLV,0.581075,1.000000,0.137587,0.640827,0.706208,0.667287,0.536022
XLE,0.182420,0.137587,1.000000,0.475634,0.226892,0.140386,0.196799
XLF,0.673378,0.640827,0.475634,1.000000,0.480688,0.553772,0.718989
XLU,0.432006,0.706208,0.226892,0.480688,1.000000,0.676913,0.465685
XLP,0.532418,0.667287,0.140386,0.553772,0.676913,1.000000,0.578305
XLY,0.835308,0.536022,0.196799,0.718989,0.465685,0.578305,1.000000


In [9]:
pcats.pca_mat

,PCA1,PCA2,PCA3,PCA4,PCA5,PCA6,PCA7
XLK,0.459628,0.235688,0.340070,0.191387,0.708588,0.137838,0.244582
XLV,0.266207,0.127809,-0.372178,-0.363693,0.325566,-0.062256,-0.729480
XLE,0.323981,-0.915040,0.039972,0.174028,0.058169,0.075150,-0.129702
XLF,0.423115,-0.076258,0.056452,-0.759291,-0.186395,-0.153887,0.420744
XLU,0.300864,0.066699,-0.698930,0.374424,-0.012955,-0.415996,0.321116
XLP,0.249183,0.123374,-0.333580,0.033324,-0.247521,0.861562,0.082130
XLY,0.530353,0.255601,0.375863,0.288075,-0.540693,-0.180154,-0.323001


In [10]:
pcats.pca_vol.to_frame()

,vol
PCA1,0.437638
PCA2,0.281855
PCA3,0.188544
PCA4,0.116934
PCA5,0.105860
PCA6,0.091051
PCA7,0.072903


In [11]:
pca_vol_year = pcats.pca_vol.to_frame()

### Out Sample Time series

In [14]:
sd = ed - datetime.timedelta(days=365 * 2)
price_df2 = yh_data.get_hist_data_for_symbols(etf_list, sd, ed, column="Close")
pca_result2 = pcats.get_pca_ts(price_df2)

In [15]:
pd.options.plotting.backend = "plotly"
pca_result2.pca_ts_df.plot()

PCA risk allocation

In [16]:
# configuration
target_pnl_vol_per_strategy = 6_000
risk_allocation = {
    "PCA1": target_pnl_vol_per_strategy,
    "PCA2": target_pnl_vol_per_strategy,
    "PCA3": target_pnl_vol_per_strategy,
    "PCA4": target_pnl_vol_per_strategy,
    "PCA5": target_pnl_vol_per_strategy,
    "PCA6": target_pnl_vol_per_strategy,
}
risk_forecast = {
    "PCA1": -0.2,
    "PCA2": 0.0,
    "PCA3": 0.0,
    "PCA4": 0.5,
    "PCA5": 1,
    "PCA6": 1,
    "PCA7": 0,
}

In [17]:
risk_allocation = pd.Series(risk_allocation, name="pnl_vol").to_frame()

risk_allocation = risk_allocation.merge(pca_vol_year, left_index=True, right_index=True)
risk_allocation["notional"] = risk_allocation["pnl_vol"] / risk_allocation["vol"]

risk_allocation = risk_allocation.merge(
    pd.Series(risk_forecast, name="forecast").to_frame(),
    left_index=True,
    right_index=True,
)

In [18]:
risk_allocation["allocation"] = risk_allocation["notional"] * risk_allocation["forecast"]
risk_allocation.round(1)

,pnl_vol,vol,notional,forecast,allocation
PCA1,6000,0.4,13710.0,-0.2,-2742.0
PCA2,6000,0.3,21287.6,0.0,0.0
PCA3,6000,0.2,31822.8,0.0,0.0
PCA4,6000,0.1,51310.9,0.5,25655.5
PCA5,6000,0.1,56678.4,1.0,56678.4
PCA6,6000,0.1,65897.0,1.0,65897.0


In [19]:
portfolio_allocation = pcats.pca_mat[risk_allocation.index].dot(risk_allocation)["allocation"].round(0)
portfolio_allocation

XLK    52895.0
XLV     4289.0
XLE    11826.0
XLF   -41345.0
XLU   -19366.0
XLP    42917.0
XLY   -36581.0
Name: allocation, dtype: float64

In [70]:
risk_allocation_test = {
    "XLE": -0_000,
    "XLF": -23_100,
    "XLI": -10_400,
    "XLK": 34_000,
    "XLU": 6_600,
    "XLV": 32_000,
    "XLP": 10_000,
    "XLB": -8_000,
    "TLT": -10_400,
    "IYR": -16_900,
}

In [71]:
risk_allocation_test = pd.Series(risk_allocation_test, name="allocationTest").to_frame()

In [72]:
risk_allocation_test["price"] = data_df.iloc[-1]
risk_allocation_test["quantity"] = risk_allocation_test["allocationTest"] / risk_allocation_test["price"]
risk_allocation_test

NameError: name 'data_df' is not defined

In [ ]:
pca_df[risk_allocation.index].T.dot(risk_allocation_test["allocationTest"]).round(0)

Portfolio History

In [ ]:
portfolio_allocation

In [ ]:
(cum_return_hist_df.dot(portfolio_allocation) / 100.0).plot()

In [ ]:
(cum_return_hist_df.dot(portfolio_allocation) / 100.0).plot()

In [ ]:
portfolio_allocation.abs().sum()

In [ ]:
portfolio_allocation.sum()

In [ ]:
400 / 27

In [ ]:
440 / 145.87

In [ ]:
397 / 271